In [3]:
from pydantic_ai import Agent

#agent = Agent(model="google-gla:gemini-2.5-flash")
agent = Agent(model="gemini-2.5-flash-lite")


#result = await agent.run("Vad är deserialisering inom programmering?")

In [ ]:
#print(result.output)

Inom programmering är **deserialisering** processen att ta data som har serialiserats (konverterats till ett format som är lätt att lagra eller överföra) och omvandla den tillbaka till sitt ursprungliga, objektorienterade format.

Tänk på det som att packa upp ett paket. När du vill använda innehållet i paketet måste du öppna det och ta ut sakerna. Deserialisering är precis vad som händer med data:

*   **Serialisering (packa):** Ett objekt (en datastruktur med egenskaper och metoder) konverteras till en sekvens av bytes eller tecken. Detta kan göras för att:
    *   **Lagra objekt:** Spara objektets tillstånd i en fil eller databas.
    *   **Överföra objekt:** Skicka objektdata över ett nätverk (t.ex. mellan en server och en webbläsare).
    *   **Interoperabilitet:** Göra data tillgänglig för andra system eller programmeringsspråk.

*   **Deserialisering (packa upp):** Den serialiserade datasekvensen tas emot och omvandlas tillbaka till ett aktivt, användbart objekt i minnet. Nu kan

In [5]:
from pydantic import BaseModel, Field

class EmployeeModel(BaseModel):
    name: str
    age: int
    salary: int = Field(gt=30000, lt=50000)
    position: str
    
result = await agent.run("Ge mig en It-anställd som arbetar i Sverige", output_type=EmployeeModel)   

print(result) 

In [ ]:
result.output.salary

49000

In [32]:
result.output.model_dump()
result.output

EmployeeModel(name='Anna', age=30, salary=49000, position='IT-specialist')

In [6]:
result = await agent.run(
    "Give me ten employees in AI and data engineering fields working in Sweden, so the roles can vary, salary must be between 30000 and 50000",
    output_type=list[EmployeeModel],
)
result

AgentRunResult(output=[EmployeeModel(name='Alice', age=30, salary=40000, position='Data Scientist'), EmployeeModel(name='Bob', age=35, salary=45000, position='Machine Learning Engineer'), EmployeeModel(name='Charlie', age=28, salary=38000, position='Data Engineer'), EmployeeModel(name='David', age=40, salary=49999, position='AI Researcher'), EmployeeModel(name='Eve', age=32, salary=42000, position='Data Analyst'), EmployeeModel(name='Frank', age=38, salary=48000, position='Software Engineer - AI'), EmployeeModel(name='Grace', age=29, salary=39000, position='Big Data Engineer'), EmployeeModel(name='Heidi', age=34, salary=46000, position='AI Specialist'), EmployeeModel(name='Ivan', age=31, salary=41000, position='Data Architect'), EmployeeModel(name='Judy', age=36, salary=47000, position='Robotics Engineer')])

In [7]:
result.output

[EmployeeModel(name='Alice', age=30, salary=40000, position='Data Scientist'),
 EmployeeModel(name='Bob', age=35, salary=45000, position='Machine Learning Engineer'),
 EmployeeModel(name='Charlie', age=28, salary=38000, position='Data Engineer'),
 EmployeeModel(name='David', age=40, salary=49999, position='AI Researcher'),
 EmployeeModel(name='Eve', age=32, salary=42000, position='Data Analyst'),
 EmployeeModel(name='Frank', age=38, salary=48000, position='Software Engineer - AI'),
 EmployeeModel(name='Grace', age=29, salary=39000, position='Big Data Engineer'),
 EmployeeModel(name='Heidi', age=34, salary=46000, position='AI Specialist'),
 EmployeeModel(name='Ivan', age=31, salary=41000, position='Data Architect'),
 EmployeeModel(name='Judy', age=36, salary=47000, position='Robotics Engineer')]

In [61]:
print(result.output[0])

name='Gustav' age=35 salary=40000 position='Data Engineer'


In [8]:
isinstance(result.output, list)

True

In [9]:
import pandas as pd

employee_list = result.output  # Lista av EmployeeModel

# Konvertera till lista av dictionaries
data = [employee.model_dump() for employee in employee_list]

df = pd.DataFrame(data)

print(df)

      name  age  salary                   position
0    Alice   30   40000             Data Scientist
1      Bob   35   45000  Machine Learning Engineer
2  Charlie   28   38000              Data Engineer
3    David   40   49999              AI Researcher
4      Eve   32   42000               Data Analyst
5    Frank   38   48000     Software Engineer - AI
6    Grace   29   39000          Big Data Engineer
7    Heidi   34   46000              AI Specialist
8     Ivan   31   41000             Data Architect
9     Judy   36   47000          Robotics Engineer


In [10]:
df_sorted = df.sort_values(by='salary', ascending=False)
print(df_sorted)

      name  age  salary                   position
3    David   40   49999              AI Researcher
5    Frank   38   48000     Software Engineer - AI
9     Judy   36   47000          Robotics Engineer
7    Heidi   34   46000              AI Specialist
1      Bob   35   45000  Machine Learning Engineer
4      Eve   32   42000               Data Analyst
8     Ivan   31   41000             Data Architect
0    Alice   30   40000             Data Scientist
6    Grace   29   39000          Big Data Engineer
2  Charlie   28   38000              Data Engineer


In [67]:
df_sorted_by_position = df.sort_values(by='position')
print(df_sorted_by_position)

     name  age  salary                     position
2   Bjorn   40   48000                AI Researcher
5    Lars   45   42000                AI Specialist
6     Eva   31   38000                 Data Analyst
0  Gustav   35   40000                Data Engineer
3    Sven   29   35000               Data Scientist
8   Freja   27   33000        Junior Data Scientist
9    Erik   42   49000             Lead AI Engineer
1  Astrid   32   45000    Machine Learning Engineer
7   Oskar   36   47000  Machine Learning Specialist
4  Ingrid   38   49000         Senior Data Engineer


In [11]:
from pydantic_ai import Tool
from typing import List

# Modell för bonus
class BonusModel(BaseModel):
    name: str
    bonus: float

@Tool
def calculate_all_bonuses(employees: List[dict]) -> List[BonusModel]:
    bonuses = []
    for emp in employees:
        bonus = emp['salary'] * 0.12 
        bonuses.append(BonusModel(name=emp['name'], bonus=bonus))
    return bonuses

agent = Agent(model="gemini-2.5-flash-lite", tools=[calculate_all_bonuses])

employee_data = df.to_dict('records')  # Konvertera DataFrame till lista av dicts

result = await agent.run(f"Beräkna bonus för dessa anställda: {employee_data}", output_type=List[BonusModel])
print(result.output)

[BonusModel(name='Alice', bonus=4800.0), BonusModel(name='Bob', bonus=5400.0), BonusModel(name='Charlie', bonus=4560.0), BonusModel(name='David', bonus=5999.88), BonusModel(name='Eve', bonus=5040.0), BonusModel(name='Frank', bonus=5760.0), BonusModel(name='Grace', bonus=4680.0), BonusModel(name='Heidi', bonus=5520.0), BonusModel(name='Ivan', bonus=4920.0), BonusModel(name='Judy', bonus=5640.0)]


In [14]:
result.output


[BonusModel(name='Alice', bonus=4800.0),
 BonusModel(name='Bob', bonus=5400.0),
 BonusModel(name='Charlie', bonus=4560.0),
 BonusModel(name='David', bonus=5999.88),
 BonusModel(name='Eve', bonus=5040.0),
 BonusModel(name='Frank', bonus=5760.0),
 BonusModel(name='Grace', bonus=4680.0),
 BonusModel(name='Heidi', bonus=5520.0),
 BonusModel(name='Ivan', bonus=4920.0),
 BonusModel(name='Judy', bonus=5640.0)]

In [16]:
from pydantic_ai import Tool

@Tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b

@Tool
def multiply_numbers(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

tool_box = [add_numbers, multiply_numbers]

agent = Agent(model="gemini-2.5-flash-lite", tools=tool_box)

result = await agent.run("Beräkna 5 plus 7")
print(f"Resultat: {result.output}")

result2 = await agent.run("Multiplicera 4 med 2")
print(f"Resultat: {result2.output}")

Traceback (most recent call last):
  File "c:\Users\henri\source\repos\Python\Objektorienterad programmering avancerad 1\ai_engineering_henrik_pilback\.venv\Lib\site-packages\pydantic_ai\models\google.py", line 483, in _generate_content
    return await func(model=self._model_name, contents=contents, config=config)  # type: ignore
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\henri\source\repos\Python\Objektorienterad programmering avancerad 1\ai_engineering_henrik_pilback\.venv\Lib\site-packages\google\genai\models.py", line 7006, in generate_content
    return await self._generate_content(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\henri\source\repos\Python\Objektorienterad programmering avancerad 1\ai_engineering_henrik_pilback\.venv\Lib\site-packages\google\genai\models.py", line 5824, in _generate_content
    response = await self._api_client.async_request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File